# 환경설정

In [ ]:
!pip install "paramiko<3.0" sshtunnel --upgrade-strategy eager --quiet

In [ ]:
from sshtunnel import SSHTunnelForwarder
import psycopg2
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import bigquery
from google.cloud import storage  # GCS 사용을 위한 라이브러리
import gspread
from google.auth import default
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
import os
import gcsfs
import io
import paramiko
import json

#인증 정보

In [ ]:
# =========================================================================
# SSH Tunnel 및 데이터베이스 접속 정보 설정 (소문자 매핑 유지)
# =========================================================================
ssh_host = "YOUR_SSH_HOST_IP"
ssh_username = "YOUR_SSH_USERNAME"
gcs_pem_path = "gs://YOUR_SECURE_BUCKET/aws-eb.pem"

db_host = "YOUR_AWS_RDS_ENDPOINT.ap-northeast-2.rds.amazonaws.com"
db_port = 5432
db_user = "YOUR_DB_READONLY_USER"
db_password = "YOUR_DB_PASSWORD"
db_name = "YOUR_DATABASE_NAME"

# =========================================================================
# GCS 직접 스트리밍을 통한 인증 키(PEM) 인-메모리 로드
# =========================================================================
try:
    # GCS에서 PEM 파일 내용을 문자열로 바로 읽어옵니다.
    fs = gcsfs.GCSFileSystem()
    with fs.open(gcs_pem_path, "rb") as f:
        pem_file_content = f.read().decode("utf-8")

    # 문자열 내용을 paramiko가 인식할 수 있는 PKey 객체로 변환합니다.
    # io.StringIO를 사용해 디스크 저장 없이 문자열을 파일처럼 다룹니다.
    pkey = paramiko.RSAKey.from_private_key(io.StringIO(pem_file_content))
    print(
        f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] GCS 인증 및 pkey 인스턴스 생성 완료"
    )
except Exception as e:
    print(f"❌ 원격 보안 키 동기화 실패: {e}")

#결제 RAW

In [ ]:


# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df = pd.read_sql("""
                         SELECT
                                    TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS r_date,
                                    TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS r_time,
                                    c.contract_uid,
                                    c.client_uid,
                                    c.actual_price,
                                    ph.price,
                                    c.status,
                                    ph.payment_status,
                                    c.product_name,
                                    pp.name product_period,
                                    p.paid_from,
                                    cl.phone_number,
                                    TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                                    TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                                    TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                                    c.is_migrated,
                                    c.is_deleted,
                                    c.initial_count,
                                    c.remain_count
                              FROM contract c
                              LEFT JOIN price_policy pp
                              ON c.price_policy_uid = pp.price_policy_uid
                              LEFT JOIN payment_history ph
                              ON c.contract_uid = ph.payment_uid
                              LEFT JOIN payment p
                              ON c.contract_uid = p.payment_uid
                              LEFT JOIN client cl
                              ON c.client_uid = cl.client_uid
                              ORDER BY ph.requested_at
                    """, conn)
    conn.close()
df.head()

In [ ]:
df['product_name'] = df['product_name'] + ' - ' + df['product_period'].astype(str)
df = df.drop(columns=['product_period'])
df.head()

In [ ]:
# 1. 'r_date' 컬럼을 날짜 데이터타입으로 변환
df['r_date'] = pd.to_datetime(df['r_date'])

# 2. '2025-06-11' 이후(포함)인 데이터만 필터링
cutoff_date = '2025-06-11'
df = df[df['r_date'] >= cutoff_date]

In [ ]:
statuses_to_exclude = ['CANCELED', 'WITHDRAW']
df_payment = df[~df['status'].isin(statuses_to_exclude)]
df_payment = df_payment[df_payment['payment_status'] == 'DONE']
df_payment = df_payment[df_payment['is_migrated'] == False]
df_payment = df_payment[df_payment['is_deleted'] == False]
df_payment = df_payment[df_payment['actual_price'] > 0]

df_payment.head()

In [ ]:
# '시간' 또는 '차감'이 포함되어 있으면 '차감권', 아니면 '기간권'
df_payment['Type'] = np.where(
    df_payment['product_name'].str.contains('주말|무제한|야간|위켄드|매일', na=False),
    '기간권',
    '차감권'
)
df_payment.head()

In [ ]:
df_차감권 = df_payment[df_payment['Type'] == '차감권']
df_기간권 = df_payment[df_payment['Type'] == '기간권']
df_기간권['product_name'].unique()

In [ ]:
df_차감권['product_name'].unique()

#기간권 전처리

In [ ]:
df_기간권 = df_기간권.dropna(subset=['actual_end_date'])
df_기간권['actual_end_date'] = pd.to_datetime(df_기간권['actual_end_date'])
df_기간권['start_date'] = pd.to_datetime(df_기간권['start_date'])
df_기간권['contract_period'] = (df_기간권['actual_end_date'] - df_기간권['start_date']).dt.days + 1
df_기간권.head()

In [ ]:
df_기간권['daily_revenue'] = df_기간권['actual_price'] / df_기간권['contract_period']

# 각 행(계약)별로 날짜 범위를 생성합니다.
df_기간권['date_range'] = df_기간권.apply(
    lambda x: pd.date_range(start=x['start_date'], end=x['actual_end_date']),
    axis=1
)

# 3. 데이터 펼치기 (Explode)
# date_range에 들어있는 날짜들을 개별 행으로 확장합니다.
df_exploded = df_기간권.explode('date_range')

# 4. 날짜 및 상품명별 그룹화
# 'date_range'와 'product_name'을 기준으로 daily_revenue를 합산합니다.
daily_summary = df_exploded.groupby(['date_range', 'product_name'])['daily_revenue'].sum().reset_index()

# 5. 컬럼명 정리 및 정렬
daily_summary.columns = ['날짜', '상품명', '일매출']
daily_summary = daily_summary.sort_values(by=['날짜', '상품명'])

# 결과 확인
daily_summary.head()

# 차감권 전처리

In [ ]:
df_차감권_종료 = df_차감권[df_차감권['status'] == 'EXPIRED']
df_차감권_사용 = df_차감권[df_차감권['status'] != 'EXPIRED']
df_차감권_사용['status'].unique()

In [ ]:
df_차감권['unit_price'] = df_차감권['actual_price'] / df_차감권['initial_count']
df_차감권_종료 = df_차감권[df_차감권['status'] == 'EXPIRED']
df_차감권_종료 = df_차감권_종료[df_차감권_종료['remain_count'] > 0]
df_차감권_종료.head()

In [ ]:

# 2. 소멸 매출액 계산 (잔여 횟수 * 회당 단가)
# 이전 단계에서 계산한 'unit_price'를 활용합니다.
df_차감권_종료['forfeiture_revenue'] = df_차감권_종료['remain_count'] * df_차감권_종료['unit_price']

# 3. 종료일 및 상품명별 그룹바이
# 이 결과는 '만료되는 날'에 한꺼번에 잡히는 매출이 됩니다.
expired_daily_summary = df_차감권_종료.groupby(['actual_end_date', 'product_name'])['forfeiture_revenue'].sum().reset_index()

# 4. 컬럼명 정리 (이전 매출 테이블들과 합치기 쉽도록 통일)
expired_daily_summary.columns = ['날짜', '상품명', '일매출액']

expired_daily_summary.sort_values(by='날짜').head()

#차감권 사용량에 따른 매출 환산

In [ ]:
# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체를 직접 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:
    # ... (이후 DB 연결 및 쿼리 실행 코드는 동일) ...
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_check_in = pd.read_sql("""
                         SELECT
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS check_in_date,
                                        ch.check_in_history_uid,
                                        ci.contract_participant_uid,
                                        c.contract_uid,
                                        cp.client_uid,
                                        ch.branch_uid,
                                        b.street_address,
                                        b.sub_type,
                                        b.display_name,
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_in_Time,
                                        TO_CHAR(ch.end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_out_time,
                                        c.product_name,
                                        c.actual_price,
                                        pp.name product_period,
                                        c.status,
                                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date
                                FROM check_in ci
                                left join check_in_history ch
                                on ci.check_in_uid = ch.check_in_uid
                                left join branch b
                                on ch.branch_uid  = b.branch_uid
                                left join contract_participant cp
                                on cp.contract_participant_uid = ci.contract_participant_uid
                                left join contract c
                                on c.contract_uid = cp.contract_uid
                                LEFT JOIN price_policy pp
                                ON c.price_policy_uid = pp.price_policy_uid
                                where c.actual_price > 0
                    """, conn)
    conn.close()
df_check_in.head()

In [ ]:
# 1. 날짜 형식 변환 (시간 제외하고 날짜만 남기기)
df_check_in['check_in_date'] = pd.to_datetime(df_check_in['check_in_date']).dt.normalize()

# 2. 지정하신 컬럼으로 그룹바이 및 유저 수(중복 제거) 집계
# 만약 '방문 횟수' 전체를 카운트해야 한다면 nunique() 대신 count()를 사용하세요.
checkin_summary = df_check_in.groupby(['check_in_date', 'contract_uid', 'product_name', 'branch_uid','sub_type']).agg({
    'client_uid': 'nunique'  # 하루에 여러 번 방문해도 유저 1명으로 집계
}).reset_index()

# 3. 컬럼명 정리

checkin_summary.head()

In [ ]:
# 1. 체크인 요약 데이터와 차감권 단가 정보 JOIN
# checkin_summary의 'contract_uid'와 df_차감권의 'contract_uid'를 기준으로 결합
df_usage_revenue = checkin_summary.merge(
    df_차감권[['contract_uid', 'unit_price']],
    on='contract_uid',
    how='left'
)

# 2. 무제한 패스 데이터 제거 (unit_price가 없는 행)
# 정기권은 이미 '기간 매출(Explode)'에서 처리되었으므로, 사용 매출 계산에서 제외합니다.
df_usage_revenue = df_usage_revenue.dropna(subset=['unit_price']).copy()

# 3. 사용 매출액(usage_revenue) 계산
# 사용자님이 관찰하신 대로 그룹바이 결과 client_uid(유저수)는 1이므로, 1 * unit_price가 됩니다.
df_usage_revenue['usage_revenue'] = df_usage_revenue['client_uid'] * df_usage_revenue['unit_price']

# 결과 확인
df_usage_revenue.head()

In [ ]:
df_usage_revenue_groupby = df_usage_revenue.groupby(['check_in_date','product_name','sub_type'])['usage_revenue'].sum().reset_index()


#제휴지점 정산

In [ ]:
df_제휴지점_raw = checkin_summary[checkin_summary['sub_type'] != 'FIVESPOT'].copy()
df_제휴지점_사용 = df_제휴지점_raw.groupby(['check_in_date','sub_type','branch_uid','product_name'])['client_uid'].sum().reset_index()
df_제휴지점_사용.head()

# 환불

In [ ]:
df_refund = df[~df['status'].isin(statuses_to_exclude)]
df_refund = df_refund[df_refund['payment_status'] == 'CANCELED']
# df_refund = df_refund[df_refund['is_migrated'] == False]
df_refund = df_refund[df_refund['is_deleted'] == False]
df_refund.head()

In [ ]:
daily_refund = df_refund.groupby(['r_date','product_name'])['price'].sum().reset_index()


#대시보드 전송

###차감형 만료

In [ ]:
# =========================================================================
# 만료 실적 데이터 타입 보정 및 정렬 (Data Type Stabilization)
# =========================================================================
df_NaN = expired_daily_summary.astype(str)

# 수치형 지표 연산을 위한 명시적 float 캐스팅 및 시계열 정렬
df_NaN["일매출액"] = df_NaN["일매출액"].astype(float)
df_NaN = df_NaN.sort_values(by="날짜", ascending=True).reset_index(drop=True)

# =========================================================================
# GCS 기반 구글 서비스 계정(Service Account) 인증 및 시트 오픈
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 바인딩 및 스프레드시트 핸들러 선언
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 회계 리포트 시트 ID 및 워크시트 지정
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("U_sales_차감형만료")

# =========================================================================
# 결측치 정제 및 멱등성 보장형 전체 덮어쓰기 (Deduplication & Overwrite)
# =========================================================================
# 결측값 공백 스트링 일괄 보정
df_for_upload = df_NaN.fillna("")

# 데이터프레임 스키마 요소를 구글 시트 업로드용 2차원 리스트 포맷으로 가공 (헤더 포함)
data_to_upload = (
    [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()
)

# 💡 gspread 호환성 보장: 최신 라이브러리 버전 버그 방지를 위해 키워드 파라미터 적용
worksheet.update(range_name="A1", values=data_to_upload)

print(
    f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 구글 스프레드시트(U_sales_차감형만료) 업데이트 성공"
)

### 기간권 매출

In [ ]:
# =========================================================================
# 기간권 매출 실적 데이터 타입 보정 및 정렬 (Data Type Stabilization)
# =========================================================================
df_NaN = daily_summary.astype(str)

# 수치형 지표 연산을 위한 명시적 float 캐스팅 및 시계열 정렬
df_NaN["일매출"] = df_NaN["일매출"].astype(float)
df_NaN = df_NaN.sort_values(by="날짜", ascending=True).reset_index(drop=True)

# =========================================================================
# GCS 기반 구글 서비스 계정(Service Account) 인증 및 시트 오픈
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 바인딩 및 스프레드시트 클라이언트 핸들러 초기화
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 회계 리포트 시트 ID 및 'U_sales' 워크시트 오픈
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("U_sales")

# =========================================================================
# 결측치 정제 및 멱등성 보장형 전체 덮어쓰기 (Deduplication & Overwrite)
# =========================================================================
# 결측값 공백 스트링 일괄 보정
df_for_upload = df_NaN.fillna("")

# 데이터프레임 스키마 요소를 구글 시트 업로드용 2차원 리스트 포맷으로 가공 (헤더 포함)
data_to_upload = (
    [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()
)

# 💡 gspread 호환성 보장: 최신 라이브러리 버전 버전 규칙에 맞추어 키워드 인자 매핑
worksheet.update(range_name="A1", values=data_to_upload)

print(
    f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 구글 스프레드시트(U_sales) 업데이트 성공"
)

### U_환불

In [ ]:
# =========================================================================
# 환불 실적 데이터 타입 보정 및 정렬 (Data Type Stabilization)
# =========================================================================
df_NaN = daily_refund.astype(str)

# 수치형 지표 연산을 위한 명시적 float 캐스팅 및 시계열 정렬
df_NaN["price"] = df_NaN["price"].astype(float)
df_NaN = df_NaN.sort_values(by="r_date", ascending=True).reset_index(drop=True)

# =========================================================================
# GCS 기반 구글 서비스 계정(Service Account) 인증 및 시트 오픈
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 바인딩 및 스프레드시트 클라이언트 핸들러 초기화
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 회계 리포트 시트 ID 및 'U_refund' 워크시트 오픈
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("U_refund")

# =========================================================================
# 결측치 정제 및 멱등성 보장형 전체 덮어쓰기 (Deduplication & Overwrite)
# =========================================================================
# 결측값 공백 스트링 일괄 보정
df_for_upload = df_NaN.fillna("")

# 데이터프레임 스키마 요소를 구글 시트 업로드용 2차원 리스트 포맷으로 가공 (헤더 포함)
data_to_upload = (
    [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()
)

# 💡 gspread 호환성 보장: 최신 라이브러리 버전 버전 규칙에 맞추어 키워드 인자 매핑
worksheet.update(range_name="A1", values=data_to_upload)

print(
    f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 구글 스프레드시트(U_refund) 업데이트 성공"
)

### 차감형 매출

In [ ]:
# =========================================================================
# 차감형 이용 실적 데이터 타입 보정 및 정렬 (Data Type Stabilization)
# =========================================================================
df_NaN = df_usage_revenue_groupby.astype(str)

# 수치형 지표 연산을 위한 명시적 float 캐스팅 및 시계열 정렬
df_NaN['usage_revenue'] = df_NaN['usage_revenue'].astype(float)
df_NaN = df_NaN.sort_values(by='check_in_date', ascending=True).reset_index(drop=True)

# =========================================================================
# GCS 기반 구글 서비스 계정(Service Account) 인증 및 시트 오픈
# =========================================================================
BUCKET_NAME = 'YOUR_SECURE_BUCKET'
KEY_FILE_IN_BUCKET = 'YOUR_SERVICE_ACCOUNT_KEY.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 바인딩 및 스프레드시트 클라이언트 핸들러 초기화
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 회계 리포트 시트 ID 및 'U_sales_차감형' 워크시트 오픈
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("U_sales_차감형")

# =========================================================================
# 결측치 정제 및 멱등성 보장형 전체 덮어쓰기 (Deduplication & Overwrite)
# =========================================================================
# 결측값 공백 스트링 일괄 보정
df_for_upload = df_NaN.fillna('')

# 데이터프레임 스키마 요소를 구글 시트 업로드용 2차원 리스트 포맷으로 가공 (헤더 포함)
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# 💡 gspread 호환성 보장: 최신 라이브러리 규칙에 맞추어 키워드 인자 명시로 버그 방지
worksheet.update(range_name='A1', values=data_to_upload)

print(f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 구글 스프레드시트(U_sales_차감형) 업데이트 성공")

### 제휴지점

In [ ]:
df_NaN = df_제휴지점_사용.astype(str)
df_NaN['client_uid'] = df_NaN['client_uid'].astype(int)
df_NaN = df_NaN.sort_values(by='check_in_date', ascending=True)

# --- 1. GCS 버킷에서 서비스 계정 정보 읽어오기 ---
# 클라우드 환경에 맞게 GCS 버킷에서 인증 정보를 가져옵니다.
BUCKET_NAME = 'fs-aws-pem'
KEY_FILE_IN_BUCKET = 'fastfive-384406-770ec5e7be85.json'

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# --- 2. gspread 인증 ---
# 읽어온 키 정보를 사용하여 인증합니다.
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# --- 3. 스프레드시트 및 워크시트 열기 ---
sheet_id = "1vakKvWwIHGn_jfwmjNuotrYOI3XgiUG4OuLdpiUzJ1E"
worksheet = gc.open_by_key(sheet_id).worksheet("df_제휴지점_사용")



# NaN 값을 빈 문자열로 바꿉니다.
df_for_upload = df_NaN.fillna('')

# --- 5. 데이터 업데이트 ---
# 데이터프레임을 리스트 형태로 변환합니다 (헤더 포함).
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# A1 셀부터 데이터를 업데이트합니다.
worksheet.update('A1', data_to_upload)

print("스프레드시트 업데이트가 성공적으로 완료되었습니다.")


In [ ]:
# =========================================================================
# 제휴지점 이용 실적 데이터 타입 보정 및 정렬 (Data Type Stabilization)
# =========================================================================
df_NaN = df_제휴지점_사용.astype(str)

# 고유 식별자 지표 연산을 위한 명시적 int 캐스팅 및 시계열 정렬
df_NaN["client_uid"] = df_NaN["client_uid"].astype(int)
df_NaN = df_NaN.sort_values(by="check_in_date", ascending=True).reset_index(
    drop=True
)

# =========================================================================
# GCS 기반 구글 서비스 계정(Service Account) 인증 및 시트 오픈
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 바인딩 및 스프레드시트 클라이언트 핸들러 초기화
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 리포트 시트 ID 및 'df_제휴지점_사용' 워크시트 오픈
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("df_제휴지점_사용")

# =========================================================================
# 결측치 정제 및 멱등성 보장형 전체 덮어쓰기 (Deduplication & Overwrite)
# =========================================================================
# 결측값 공백 스트링 일괄 보정
df_for_upload = df_NaN.fillna("")

# 데이터프레임 스키마 요소를 구글 시트 업로드용 2차원 리스트 포맷으로 가공 (헤더 포함)
data_to_upload = (
    [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()
)

# 💡 gspread 호환성 보장: 최신 라이브러리 규칙에 맞추어 키워드 인자 명시로 버그 방지
worksheet.update(range_name="A1", values=data_to_upload)

print(
    f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 구글 스프레드시트(df_제휴지점_사용) 업데이트 성공"
)